In [1]:
import pandas as pd

In [2]:
# read all excel files in a path

import os
import glob

os.chdir(os.path.dirname(os.getcwd()))

path = r'data/final/tables/annotations/filled' # use your path
all_files = glob.glob(os.path.join(path, "*.xlsx"))

df_from_each_file = (pd.read_excel(f, skiprows = 3) for f in all_files)
df = pd.concat(df_from_each_file, ignore_index=True)

c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [3]:
df.dropna(subset=['id'], inplace=True)

In [4]:
# Keeping only evaluation of existing rows 

df['id'] = df['id'].astype(str)
df['id_general'] = df['id'].str.replace(r'X_', '', regex=True)

In [5]:
def check_correct_result(df):
    """
    Groups by 'id' and checks for the following conditions:
    - If at least one 'Ja' and at least one 'Nein' exists -> 'failed to extract all correct metrics'
    - If at least one 'Ja' exists -> 'extracted all correct metrics'
    - If all values are NaN -> 'no extracted metric'
    - Otherwise -> 'incorrect metric'

    Args:
    df (pd.DataFrame): The input DataFrame.

    Returns:
    pd.DataFrame: A DataFrame with 'id' and 'correct_result'.
    """
    def determine_result(group):
        has_ja = group['Wert korrekt? (Ja/ Nein)'].eq('Ja').any()
        has_nein = group['Wert korrekt? (Ja/ Nein)'].eq('Nein').any()
        all_na = group['Wert korrekt? (Ja/ Nein)'].isna().all()

        if has_ja and has_nein:
            return '3. Failed extraction: LLM failed to extract all of the metrics correctly'
        elif has_ja:
            return '1. Correct extraction: LLM extracted all metrics correctly'
        elif has_nein:
            return '4. Failed extraction: LLM failed to extract any of the metrics correctly'
        elif all_na:
            return '2. Correct extraction: no extracted metric'

    # Group by 'id' and apply the function
    result_df = df.groupby('id_general').apply(determine_result).reset_index(name='correct_result')
    
    return result_df


In [6]:
def evaluate_llm_performance_on_data(df):

    metrics_evaluation = []

    for metric in df['met'].unique():

        keyword_subset = df[df['met'] == metric].copy()  #

        keyword_subset['value_match'] = keyword_subset['Wert korrekt? (Ja/ Nein)'].apply(lambda x: 1 if x == 'Ja' or pd.isna(x) else 0)

        evaluation_results = check_correct_result(keyword_subset).value_counts('correct_result').reset_index()

        evaluation_results['metric'] = metric

        metrics_evaluation.append(evaluation_results)
    
    return pd.concat(metrics_evaluation, axis=0)
    

In [7]:
evaluate_data = evaluate_llm_performance_on_data(df)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21680\13119520.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result_df = df.groupby('id_general').apply(determine_result).reset_index(name='correct_result')
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_21680\13119520.py:30: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result_df = df.groupby('id_general').apply(determine_result).reset_index(name='correct_r

In [8]:
evaluate_data_pivot = evaluate_data.pivot(index='correct_result', columns='metric', values='count').fillna(0)

In [9]:
evaluate_data_pivot.loc["Total"] = evaluate_data_pivot.sum()


evaluate_data_pivot

In [13]:
evaluate_data_pivot

metric,eg_fok_unit,eg_fok_value,fok_unit,fok_value,gfz_value,gok_unit,gok_value,grundwasser_value,grz_value,hw100_value,hw10_value
correct_result,,,,,,,,,,,
1. Correct extraction: LLM extracted all metrics correctly,3.0,4.0,0.0,0.0,6.0,2.0,2.0,2.0,10.0,1.0,0.0
2. Correct extraction: no extracted metric,52.0,52.0,59.0,59.0,35.0,58.0,58.0,55.0,31.0,57.0,60.0
3. Failed extraction: LLM failed to extract all of the metrics correctly,1.0,2.0,1.0,1.0,3.0,0.0,0.0,1.0,3.0,1.0,0.0
4. Failed extraction: LLM failed to extract any of the metrics correctly,5.0,3.0,1.0,1.0,17.0,1.0,1.0,3.0,17.0,2.0,1.0
Total,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0,61.0


In [10]:
evaluate_data_pivot[['eg_fok_unit',	'eg_fok_value',	'fok_unit',	'fok_value', 'gok_unit', 'gok_value']]

metric,eg_fok_unit,eg_fok_value,fok_unit,fok_value,gok_unit,gok_value
correct_result,,,,,,
1. Correct extraction: LLM extracted all metrics correctly,3.0,4.0,0.0,0.0,2.0,2.0
2. Correct extraction: no extracted metric,52.0,52.0,59.0,59.0,58.0,58.0
3. Failed extraction: LLM failed to extract all of the metrics correctly,1.0,2.0,1.0,1.0,0.0,0.0
4. Failed extraction: LLM failed to extract any of the metrics correctly,5.0,3.0,1.0,1.0,1.0,1.0
Total,61.0,61.0,61.0,61.0,61.0,61.0


In [11]:
evaluate_data_pivot[['grz_value', 'gfz_value']]

metric,grz_value,gfz_value
correct_result,,
1. Correct extraction: LLM extracted all metrics correctly,10.0,6.0
2. Correct extraction: no extracted metric,31.0,35.0
3. Failed extraction: LLM failed to extract all of the metrics correctly,3.0,3.0
4. Failed extraction: LLM failed to extract any of the metrics correctly,17.0,17.0
Total,61.0,61.0


In [12]:
evaluate_data_pivot[['grundwasser_value', 'hw100_value','hw10_value']]

metric,grundwasser_value,hw100_value,hw10_value
correct_result,,,
1. Correct extraction: LLM extracted all metrics correctly,2.0,1.0,0.0
2. Correct extraction: no extracted metric,55.0,57.0,60.0
3. Failed extraction: LLM failed to extract all of the metrics correctly,1.0,1.0,0.0
4. Failed extraction: LLM failed to extract any of the metrics correctly,3.0,2.0,1.0
Total,61.0,61.0,61.0


In [14]:
from tabulate import tabulate

print(tabulate(evaluate_data_pivot, tablefmt="pipe", headers="keys"))

| correct_result                                                           |   eg_fok_unit |   eg_fok_value |   fok_unit |   fok_value |   gfz_value |   gok_unit |   gok_value |   grundwasser_value |   grz_value |   hw100_value |   hw10_value |
|:-------------------------------------------------------------------------|--------------:|---------------:|-----------:|------------:|------------:|-----------:|------------:|--------------------:|------------:|--------------:|-------------:|
| 1. Correct extraction: LLM extracted all metrics correctly               |             3 |              4 |          0 |           0 |           6 |          2 |           2 |                   2 |          10 |             1 |            0 |
| 2. Correct extraction: no extracted metric                               |            52 |             52 |         59 |          59 |          35 |         58 |          58 |                  55 |          31 |            57 |           60 |
| 3. Failed extracti

| correct_result                                                           |   eg_fok_unit |   eg_fok_value |   fok_unit |   fok_value |   gfz_value |   gok_unit |   gok_value |   grundwasser_value |   grz_value |   hw100_value |   hw10_value |
|:-------------------------------------------------------------------------|--------------:|---------------:|-----------:|------------:|------------:|-----------:|------------:|--------------------:|------------:|--------------:|-------------:|
| 1. Correct extraction: LLM extracted all metrics correctly               |             3 |              4 |          0 |           0 |           6 |          2 |           2 |                   2 |          10 |             1 |            0 |
| 2. Correct extraction: no extracted metric                               |            52 |             52 |         59 |          59 |          35 |         58 |          58 |                  55 |          31 |            57 |           60 |
| 3. Failed extraction: LLM failed to extract all of the metrics correctly |             1 |              2 |          1 |           1 |           3 |          0 |           0 |                   1 |           3 |             1 |            0 |
| 4. Failed extraction: LLM failed to extract any of the metrics correctly |             5 |              3 |          1 |           1 |          17 |          1 |           1 |                   3 |          17 |             2 |            1 |
| Total                                                                    |            61 |             61 |         61 |          61 |          61 |         61 |          61 |                  61 |          61 |            61 |           61 |